In [1]:
from langgraph.graph import StateGraph
from typing import TypedDict
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from chat_memory import get_chat_history
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage
from video_processing import analyze_video
from vectorstore_setup import clean_classification_text
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langgraph.graph import StateGraph, START, END
import tempfile

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="path/to/your/chroma_db", embedding_function=embeddings)

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")



# when a query comes in, use this model to embed the query text so you can compare it against the vectors already stored.
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = Chroma(persist_directory="/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/chroma_db", 
                                    embedding_function=embeddings)


# Whisper speach to text 
from openai import OpenAI
client = OpenAI()




In [2]:
router_prompt = ChatPromptTemplate.from_messages([
    ("system", """Your job is to classify user queries into two buckets: 
     - memory
     - memory and vectorstore
     
     *Guidelines*
     Questions regarding exercises, exercise form, or technique: vectorstore & memory
     General question or statements: memory
   
     *Respond to queries with only the correct class*
     Examples: 
     User: how should I grip the bar for bench press?
     Agent: vectorstore & memory

     User: what muscles should I feel during the Romanian dead lift?
     Agent: vectorstore & memory

     User: How's this? 
     Agent: vectorstore & memory

     User: Is this good squat technique? 
     Agent: vectorstore & memory

     User: Was my bar path better?
     Agent: vectorstore & memory

     User: thakns for your help!
     Agent: memory

"""),

     ("human", "{query}")
     
])

llm = ChatOpenAI(model='gpt-4o')

output_parser = StrOutputParser()

router_chain = router_prompt | llm | output_parser


In [3]:
# Define the state

# the state is a dictionary

class GraphState(TypedDict):

    session_id: int 

    user_query: str

    user_video: str

    classification_image: str

    classified_keywords: str

    top_k_chunks: str

    encoded_images: list[str]

    response: str


workflow = StateGraph(GraphState) 

In [4]:
def video_encoder_node(state: GraphState):
    user_video = state["user_video"]
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=20, max_seconds=10) # encodes video
    encoded_images = [] # creates a list of encoded images for LLM
    for images in user_video_encoded:
        encoded_images.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{images}"}})
        classification_image = user_video_encoded[0] 
    
    return {"classification_image": classification_image, "encoded_images": encoded_images}


In [5]:
# video classification router NODE

def video_classification_node(state: GraphState):
    classification_image = state["classification_image"]

    router_llm = ChatOpenAI(model='gpt-4o')

    response = router_llm.invoke([

    {"role": "user", "content": 

    [{"type": "text", "text":  """Your job is to analyze images of users working out for proper form, and list the key checkpoints of their to body evaluate. 
    Give me ONLY the bodypart checkpoints. Do NOT include evaluation suggestions. Do NOT include an intro sentence. 
    Output format should be exactly the example below.
    **Example**
    Overhead press

    1. Feet & base
    2. Glutes & legs
    3. Core & Ribcage
    4. Shoulder position
    5. Bar path
    6. Head & Neck
    7. Lockout position
    8. Tempo and control
    """},


        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{classification_image}"}}
        ]}


    ])

    classified_keywords = clean_classification_text(response)

    return {"classified_keywords": classified_keywords}


In [6]:
def vector_db_node(state: GraphState):
    user_query = state["user_query"]
    classified_keywords = state.get("classified_keywords", "")
    retrieval_query = classified_keywords

    if classified_keywords:
        results_user = vectorstore.similarity_search(user_query, k=5)
        results_video = vectorstore.similarity_search(retrieval_query, k=5)
        results = results_user + results_video

    if not classified_keywords: 
        results_user = vectorstore.similarity_search(user_query, k=5)
        results = results_user

    unique_results = []
    seen = set()

    for r in results:
        content = r.page_content
        if content not in seen:
            seen.add(content)
            unique_results.append(r)


    top_k_chunks = "\n".join([r.page_content for r in unique_results])

    return {"top_k_chunks": top_k_chunks}

In [7]:
def response_generator(state: GraphState):
    top_k_chunks = state['top_k_chunks']
    encoded_images = state.get("encoded_images", [])
    session_id = state["session_id"]
    user_query = state["user_query"]
    classified_keywords = state["classified_keywords"]

    prompt = ChatPromptTemplate.from_messages([
            ("system", """
             You are a world-class fitness coach. You are positive and uplifting. You love helping users achieve their fitness goals.
             Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 
    Inspect each image CLOSELY and carefully for problems or issues related to best practices in exercise form. Help the user diagnose their incorrect form. 
    
        # Use these {classified_keywords} to help diagnose any issues. 
        # If there are no issues with the checkpoint, SKIP IT and move on.
        # Resopnd with only the 1 to 3 most important checkpoints MAXIUMUM. Do not overwhelm users with information.
        # If important checkpoints are hidden due to equipment or camera angle, say so.
        # Respond conversationally, as a coach would.
        # Don't nitpick or overcorrect. 
             
    ---
        # ANSWER CONTEXT
    Use ONLY the following context when answering a user: 
 
    {top_k_chunks}
    
    If the query or image isn't in context, reply, 'I don't have expert coaching content for this exercise yet. 
    Currently I can analyze: bench press, overhead press, incline bench press...'"
        
    ---  


    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])



    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query},
        *encoded_images,   # <- your list of {"type":"image_url",...}
    ])

    llm = ChatOpenAI(model='gpt-5',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg], "top_k_chunks": top_k_chunks, "classified_keywords": classified_keywords},
        config={"configurable": {"session_id": session_id}}
    )


    return {"response": response}
    

In [8]:
def chat_memory(state: GraphState):
    user_query = state["user_query"]
    session_id = state["session_id"]
    
    prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a world-class fitness coach. You have extensive experience in helping weight lifters achieve perfect form and maximum hypertrophy. 
    Your job is to analyze images of users lifting weights, offer them advice from your context, and to answer any questions they might have. 


    """),
        MessagesPlaceholder(variable_name="history"),
        MessagesPlaceholder(variable_name="input"),
    ])

    user_msg = HumanMessage(content=[
        {"type": "text", "text": user_query}
    ])

    llm = ChatOpenAI(model='gpt-4o',
                    temperature=0.5)

    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    chain_with_history = RunnableWithMessageHistory(
        chain,
        get_session_history=get_chat_history,
        input_messages_key="input",
        history_messages_key="history"

    )

    response = chain_with_history.invoke(
        {"input": [user_msg]},
        config={"configurable": {"session_id": session_id}}
    )

    return {"response": response}
    


# Router function

In [9]:
def route_query(state: GraphState):
    # first check: is there a video?
    if state["user_video"]:
        return "video_encoder"
    
    # no video — ask the LLM which path
    route = router_chain.invoke({"query": state["user_query"]})
    route = route.lower().strip()
    
    if "vectorstore" in route:
        return "vector_db"
    else:
        return "chat_memory"

# Audio transcription function

In [10]:
# def transcribe_audio(audio_path):
#     audio_file = open(audio_path, "rb") # rb = read binary. audio files are binary data (raw bytes). this tells python gto open as binary instaed of text
#     transcription = client.audio.transcriptions.create(
#         model="whisper-1",
#         file=audio_file
#     )

#     return transcription.text
    

# user_query = transcribe_audio("/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_voice_notes/bench_press_voice_note.m4a")
# print(user_query)

In [11]:
# conditional edge
workflow.add_conditional_edges(START, route_query)

# add nodes
workflow.add_node("video_encoder", video_encoder_node)
workflow.add_node("video_classification_router", video_classification_node)
workflow.add_node("vector_db", vector_db_node)
workflow.add_node("response_generator", response_generator)
workflow.add_node("chat_memory", chat_memory)



# connect them
workflow.add_edge("chat_memory", END)
workflow.add_edge("video_encoder", "video_classification_router")
workflow.add_edge("video_classification_router", "vector_db")
workflow.add_edge("vector_db", "response_generator")
workflow.add_edge("response_generator", END)

app = workflow.compile()


In [12]:
result = app.invoke({
    "session_id": 234,
    "user_query": "how's my form?",
    "user_video": "/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/raw_workout_videos/bench_press/bench_07.mov"
})

print(result["response"])

Frames processed: 20, (10s cap)
Saved to processed-images/bench_07
Video name: bench_07
Great setup and control. A couple key fixes will make it stronger:

1) Hips/glutes: Your butt pops off the bench on the press. Keep hips down so glutes stay in contact the whole rep. Cue: squeeze your glutes to the pad and drive your heels “through the floor” so the leg drive goes along the bench into your arched upper back, not up into a bridge.

2) Feet/leg position: Your base looks a bit unstable. Plant both heels flat and bring your feet back until your shins are nearly vertical, legs close to the bench. Keep that position set so they don’t move between reps.

3) Elbow angle/bar path: Elbows look a bit flared. On the descent, tuck them to about 45 degrees to your torso and fix your eyes on one ceiling point—touch, then drive the bar back toward that same point for a consistent path.

Try a few sets of five with lighter weight to groove these cues, then build back up. You’ve got this.


In [13]:

# result = app.invoke({
#     "session_id": 123,
#     "user_query": user_query,
#     "user_video": ""
# })

# print(result["response"])

In [14]:
# result = app.invoke({
#     "session_id": 123,
#     "user_query": "thanks for your help today!",
#     "user_video": ""
# })

# print(result["response"])

In [15]:
# result = app.invoke({
#     "session_id": 123,
#     "user_query": "can you tell me again how to make it easier?",
#     "user_video": ""
# })

# print(result["response"])

In [19]:
def correctness_evaluator(run, example):
    generated = run.outputs["answer"]
    reference = example.outputs["answer"]
    question = example.inputs["user_query"]

    prompt = f"""You are evaluating a fitness coaching AI assistant.
Your job is to assess the quality of its response against a reference answer.

Question: {question}
Reference answer: {reference}
Generated answer: {generated}

Scoring criteria:

Score 0 — The generated answer contains incorrect or potentially harmful advice that contradicts the reference answer. This is the worst outcome.
Score 1 — The generated answer is incomplete or omits important details from the reference, but contains nothing incorrect or harmful.
Score 2 — The generated answer is correct and captures the key facts from the reference answer.

Important: Penalize incorrect advice more harshly than omission. A response that says nothing is better than one that says something wrong.

Reply with only the number: 0, 1, or 2."""

    from openai import OpenAI
    openai_client = OpenAI()
    result = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    try: 
        score_raw = result.choices[0].message.content.strip()
        score = int(''.join(filter(str.isdigit, score_raw))[0])
    except:
        score = -1
    return {"key": "correctness", "score": score / 2} # normalize the score so it's <= 1

In [21]:
def run_rag_pipeline(inputs: dict, attachments: dict) -> dict:
    # grab video from attachments in LangSmith
    video_name = list(attachments.keys())[0] 
    # read the video as raw bytes
    video_bytes = attachments[video_name]["reader"].read()
    
    # save to temp file because analyze_video expects a filepath
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        # . delete=False means "don't delete this file when we're done writing" because we still need it for the pipeline.
        tmp.write(video_bytes)
        tmp_path = tmp.name
    
    state = {
        "user_query": inputs["user_query"],
        "session_id": "eval-session",
        "classified_keywords": "",
        "encoded_images": [],
        "user_video": tmp_path
    }
    
    state = {**state, **video_encoder_node(state)}
    state = {**state, **video_classification_node(state)}
    state = {**state, **vector_db_node(state)}
    state = {**state, **response_generator(state)}

    print("STATE KEYS:", state.keys())
    print("RESPONSE:", state.get("response", "NOT FOUND"))
    
    return {"answer": state["response"]}

In [22]:
from langsmith import Client
from langsmith.evaluation import evaluate

langsmith_client = Client()

os.environ["LANGCHAIN_TRACING_V2"] = "false" 
# stops LangSmith from trying to log all the big intermediate data
# evaluate() function will still capture your inputs, outputs, and evaluator scores
# it just won't try to upload the full trace with all those heavy frames.

results = evaluate(
    run_rag_pipeline,
    data="Image assessment response accuracy v1",
    evaluators=[correctness_evaluator],
    experiment_prefix="vision-faithfullness-rag-v4-short-answers"
)

View the evaluation results for experiment: 'vision-faithfullness-rag-v4-short-answers-3a78cd9d' at:
https://smith.langchain.com/o/5dffb1fc-82e5-4000-a9a5-d9c923e0f73c/datasets/1be4093e-6f7d-4021-a94b-d6cf76c22b6a/compare?selectedSessions=4cc7d3dd-4e33-4452-8932-60f2e7883476




0it [00:00, ?it/s]

Frames processed: 20, (10s cap)
Saved to processed-images/tmps5i2ef93
Video name: tmps5i2ef93
STATE KEYS: dict_keys(['user_query', 'session_id', 'classified_keywords', 'encoded_images', 'user_video', 'classification_image', 'top_k_chunks', 'response'])
RESPONSE: You’re moving the weight well! Two quick fixes will make it feel stronger and safer:

- Feet/leg drive: Your heels look elevated. Plant your feet flat and slide them back as far as you comfortably can while staying flat. Keep them glued to the floor and squeeze glutes to create a solid base.

- Bar path + elbows: You’re touching high near the collarbone with flared elbows. Lower to the lower chest with elbows about 45 degrees to your torso, then press back toward the same point on the ceiling your eyes are fixed on.

Note: Hard to see wrists from this angle, so I can’t confirm grip/wrist stack.
Frames processed: 20, (10s cap)
Saved to processed-images/tmpcoke2dx1
Video name: tmpcoke2dx1
STATE KEYS: dict_keys(['user_query', 'ses

Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:2426)')))"))
Content-Length: 19811221
API Key: lsv2_********************************************cctrace=019ce211-31ab-7781-85a2-c6a24b132ec6,id=019ce211-3719-7933-a18a-c542293314b6
Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. The content length of 39617714 bytes exceeds the maximum size limit of 20971520 bytes. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:242

STATE KEYS: dict_keys(['user_query', 'session_id', 'classified_keywords', 'encoded_images', 'user_video', 'classification_image', 'top_k_chunks', 'response'])
RESPONSE: Strong reps! Two quick fixes to make your dumbbell bench feel tighter and safer:

- Upper back set: Pinch your shoulder blades together and keep them pinned to the bench the whole set. Chest up, shoulders down—this protects the shoulders and gives you a stable platform.

- Elbow angle + DB path: You’re flaring a bit. On the way down, keep elbows tucked ~30–45° to your torso with the bells stacked over your elbows. Press up over the shoulders without clanking the DBs together at the top.

If your hips are lifting, plant your feet and keep glutes on the bench while driving through the floor. Angle hides your wrists, so just keep them neutral under the bells.
Frames processed: 20, (10s cap)
Saved to processed-images/tmpix417yu7
Video name: tmpix417yu7


...

Error running target function: [Errno 2] No such file or directory: '/Users/chandlershortlidge/Desktop/Ironhack/fitness-form-coach/data/processed/processed-images/tmp8pan3toy_19.jpg'
Traceback (most recent call last):
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/evaluation/_runner.py", line 1903, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/opt/miniconda3/envs/fitness-form-coach/lib/python3.10/site-packages/langsmith/run_helpers.py", line 758, in wrapper
    function_result = run_container["context"].run(
  File "/var/folders/2n/c95ntf5d1wx1_vrd1b3ljbkw0000gn/T/ipykernel_60682/719908470.py", line 21, in run_rag_pipeline
    state = {**state, **video_encoder_node(state)}
  File "/var/folders/2n/c95ntf5d1wx1_vrd1b3ljbkw0000gn/T/ipykernel_60682/3309625947.py", line 3, in video_encoder_node
    user_video_encoded = analyze_video(filepath_in=user_video, frame_count=20, max_seconds=10) # encodes video
  File "/Users/chandlersho

Frames processed: 20, (10s cap)
Saved to processed-images/tmp8pan3toy
Video name: tmp8pan3toy
Frames processed: 20, (10s cap)
Saved to processed-images/tmpz2axyc6s
Video name: tmpz2axyc6s


Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. The content length of 23781378 bytes exceeds the maximum size limit of 20971520 bytes. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:2426)')))"))
Content-Length: 23781378
API Key: lsv2_********************************************cctrace=019ce212-7349-7742-a21a-cf1cf8fc4493,id=019ce212-83ee-7fd0-9aad-bc0ef2ec3d4e; trace=019ce212-7349-7742-a21a-cf1cf8fc4493,id=019ce214-2774-7a22-bdb3-ae4b99a5e390
Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. The content length of 23782243 bytes exceeds the maximum size limit of 20971520 bytes. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchai

STATE KEYS: dict_keys(['user_query', 'session_id', 'classified_keywords', 'encoded_images', 'user_video', 'classification_image', 'top_k_chunks', 'response'])
RESPONSE: Strong pressing! Here are the two biggest wins for you:

- Bar path + elbows: You’re touching a bit high with some flare. Lower the bar to your lower chest with elbows about 45 degrees to your torso, then “throw it back” toward the point on the ceiling your eyes are fixed on (slight upside‑down J path).

- Feet/leg drive: Your right heel pops and the foot shifts. Pick one style and keep it planted—either heels flat or stay on your toes—but keep both feet glued to the floor, knees out a touch, and squeeze your glutes to create a solid base.

Note: Your wrists look like they’re bending back. Crush the bar and keep knuckles to the ceiling so the bar stacks over your forearms.


Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. The content length of 26464886 bytes exceeds the maximum size limit of 20971520 bytes. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /runs/multipart (Caused by SSLError(SSLError(5, '[SYS] unknown error (_ssl.c:2426)')))"))
Content-Length: 26464886
API Key: lsv2_********************************************cctrace=019ce214-342a-7570-9187-54602a66d3d9,id=019ce214-3f5d-7273-8594-3aa7025269ab; trace=019ce214-342a-7570-9187-54602a66d3d9,id=019ce216-5c2d-7d13-9ce2-bb7496e8d5fa
Failed to send compressed multipart ingest: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. The content length of 26455996 bytes exceeds the maximum size limit of 20971520 bytes. SSLError(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchai